# Step 21 — ADP-A Cloud Training (Qwen3-4B QLoRA)
**Phase:** 4.1 — Cloud Retraining
**Spec authority:** SPEC-400 §3.2; REQ-400-030–045
**Platform:** Lightning.ai A10G (24 GB VRAM)
**Base model:** `Qwen/Qwen3-4B`
**Prerequisites:** Step 20 complete (`finetuning/adp_a_empathy/data/adp_a_cloud_train.jsonl`)
**Output adapter:** `finetuning/adp_a_empathy/adp_a_cloud_final/` → pushed to HF Hub

---

## Why Qwen3-4B (not Phi-3.5-mini)

Qwen3-4B replaces Phi-3.5-mini as ADP-A for three reasons:
1. **Already in the production pipeline** — Qwen3-4B is the base model for the
   `/pipeline` endpoint's pre-analysis pass (Step 1.5). Loading it once for both
   pre-analysis and ADP-A eliminates a second full model load.
2. **Strong multilingual and emotional reasoning** — Qwen3's instruction-following
   and conversational empathy outperform Phi-3.5 on mental-health benchmarks.
3. **Thinking mode toggleable** — `enable_thinking=False` disables chain-of-thought
   for classification tasks (ADP-B/C); for ADP-A generative tasks, the default
   mode is appropriate.

## QLoRA on A10G

`BitsAndBytesConfig(load_in_4bit=True)` is safe on Lightning.ai — the A10G has
a persistent CUDA context, unlike HF ZeroGPU where `bitsandbytes` crashes at
import time. 4-bit quantisation reduces Qwen3-4B from ~8 GB to ~4 GB, leaving
~20 GB free for activations and gradient buffers at larger batch sizes.

## A10G-specific hyperparameter changes vs RTX 3070

| Parameter | Local (RTX 3070) | Cloud (A10G) | Reason |
|-----------|-----------------|-------------|--------|
| `BATCH_SIZE` | 2 | **8** | 3× VRAM headroom |
| `GRAD_ACCUM` | 4 | **2** | Effective batch 16 maintained |
| `MAX_SEQ_LENGTH` | 768 | **1024** | Multi-turn examples up to 10 turns fit |
| `NUM_EPOCHS` | 5 | **5** | Same convergence target |
| Quantisation | None (bf16) | **NF4 4-bit** (QLoRA) | Frees VRAM for larger batch |
| `report_to` | none | **wandb** (optional) | Lightning.ai integrates W&B |


In [ ]:
# ── Cell 0: Environment setup ─────────────────────────────────────────────────
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN)
    print("HF Hub authenticated.")
else:
    print("WARNING: HF_TOKEN not set — cannot push adapter to Hub after training.")

import torch
print(f"Device : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


In [ ]:
# ── Cell 1: Configuration ────────────────────────────────────────────────────
import json, gc
from pathlib import Path

BASE_MODEL_ID = "Qwen/Qwen3-4B"
HF_OUTPUT_REPO = "equinox013/nikko-adp-a"   # push destination — update to your repo

BASE_DIR       = Path("/teamspace/studios/this_studio/nikko-companion")
FINETUNING     = BASE_DIR / "finetuning"
TRAIN_DATA     = FINETUNING / "adp_a_empathy" / "data" / "adp_a_cloud_train.jsonl"
CHECKPOINT_DIR = FINETUNING / "adp_a_empathy" / "cloud_checkpoints"
OUTPUT_DIR     = FINETUNING / "adp_a_empathy" / "adp_a_cloud_final"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert TRAIN_DATA.exists(), f"Training data not found: {TRAIN_DATA} — run Step 20 first."

# ── A10G hyperparameters ──────────────────────────────────────────────────────
# A10G 24 GB vs RTX 3070 8 GB: 3× VRAM headroom allows larger batch + longer
# sequences. Effective batch = BATCH_SIZE × GRAD_ACCUM = 16 maintained.
NUM_EPOCHS     = 5
BATCH_SIZE     = 8       # up from 2 (A10G headroom)
GRAD_ACCUM     = 2       # effective batch = 16
MAX_SEQ_LENGTH = 1024    # up from 768; accommodates 10-turn multi-turn examples
LEARNING_RATE  = 1e-4
WEIGHT_DECAY   = 0.01

# ── QLoRA / LoRA configuration ────────────────────────────────────────────────
# [CONCEPT] QLoRA (Quantisation-aware LoRA)
# Standard QLoRA: load the base model in 4-bit NF4 (via BitsAndBytesConfig),
# inject LoRA adapters in fp32. During the backward pass, activations are
# temporarily upcast to bf16 for gradient computation. This allows training a
# 4B model on 4 GB VRAM instead of 8 GB, leaving room for larger batches.
# On Lightning.ai A10G: bitsandbytes works normally (persistent CUDA context).
# On HF ZeroGPU: bitsandbytes crashes at import time — do NOT use there.
LORA_R        = 16
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.05
# Qwen3 attention and MLP projection module names (same as Qwen2)
LORA_TARGETS  = ["q_proj", "k_proj", "v_proj", "o_proj",
                 "gate_proj", "up_proj", "down_proj"]

print(f"Base model     : {BASE_MODEL_ID}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"HF push repo   : {HF_OUTPUT_REPO}")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM}  |  MaxSeqLen: {MAX_SEQ_LENGTH}")


## Section 1 — Dataset Loading & Formatting

Qwen3 uses its own chat template via `apply_chat_template()`. The `/no_think`
system token disables thinking mode for the fine-tuning pass — we want
direct generation, not CoT preambles.

A stratified 90/10 split by approximate instruction length maintains length
distribution in both train and eval splits.


In [ ]:
# ── Cell 2: Dataset loading ───────────────────────────────────────────────────
import random
from collections import defaultdict
from datasets import Dataset
from transformers import AutoTokenizer

random.seed(42)

raw_records = [
    json.loads(line)
    for line in TRAIN_DATA.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
print(f"Loaded {len(raw_records)} records.")

# ── Tokenizer + chat template ─────────────────────────────────────────────────
# trust_remote_code=True is required for Qwen3's custom tokenizer class.
# Qwen3 chat template: no explicit /no_think token needed for SFT — the model
# will be in instruction-following mode throughout training.
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # left-padding corrupts SFTTrainer loss masking

def format_record(record: dict) -> dict:
    # [CONCEPT] Qwen3 chat template
    # Qwen3 uses <|im_start|>role\ncontent<|im_end|> formatting.
    # apply_chat_template() handles this natively — same API as Gemma-2 and Phi-3.
    # We include add_generation_prompt=False so the assistant turn is included
    # in the formatted string and contributes to the loss (not just the prompt).
    messages = [
        {"role": "user",      "content": record["instruction"]},
        {"role": "assistant", "content": record["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}

# ── 90/10 split stratified by record length bucket ───────────────────────────
# Stratify by rough length (short/medium/long) to keep the distribution
# of multi-turn (longer) and single-turn (shorter) records even across splits.
def length_bucket(r):
    n = len(r["instruction"].split()) + len(r["output"].split())
    return "short" if n < 80 else "medium" if n < 200 else "long"

buckets = defaultdict(list)
for r in raw_records:
    buckets[length_bucket(r)].append(r)

train_records, eval_records = [], []
for _, items in buckets.items():
    random.shuffle(items)
    n_eval = max(1, int(len(items) * 0.10))
    eval_records.extend(items[:n_eval])
    train_records.extend(items[n_eval:])

random.shuffle(train_records)
random.shuffle(eval_records)

train_data = Dataset.from_list([format_record(r) for r in train_records])
eval_data  = Dataset.from_list([format_record(r) for r in eval_records])
print(f"Train: {len(train_data)}  |  Eval: {len(eval_data)}")

long_count = sum(1 for r in train_data if len(r["text"]) > MAX_SEQ_LENGTH * 4 * 0.8)
if long_count:
    print(f"Records near seq limit: {long_count} — will be truncated at {MAX_SEQ_LENGTH} tokens.")


## Section 2 — Model Loading (QLoRA, 4-bit NF4)

`BitsAndBytesConfig` is safe on Lightning.ai — persistent CUDA context.
Qwen3-4B at 4-bit: ~4 GB. Total peak VRAM during training: ~10–14 GB.
`prepare_model_for_kbit_training()` is required for NF4 bases (unlike bf16
where it's a no-op).


In [ ]:
# ── Cell 3: Model load (QLoRA NF4) ───────────────────────────────────────────
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

gc.collect()
torch.cuda.empty_cache()

# [CONCEPT] BitsAndBytesConfig NF4
# NF4 (Normal Float 4) is a data type specifically designed for QLoRA training.
# Each weight is stored in 4 bits using a quantisation map optimised for the
# normal distribution of neural network weights. During the forward pass,
# weights are dequantised to bf16 on the fly. During backward, gradients
# flow through the LoRA matrices (in fp32) — the base weights are never updated.
# Net effect: ~8 GB Qwen3-4B (bf16) → ~4 GB (NF4), freeing ~4 GB for batch/activations.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,   # second quantisation of the quantisation constants
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {BASE_MODEL_ID} in NF4 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,   # required for Qwen3 custom model class
)
model.config.use_cache = False

# [CONCEPT] prepare_model_for_kbit_training
# When the base model is in NF4/int8, this function upcasts layer norms
# to fp32 (for numerical stability during training) and enables gradient
# checkpointing. This is specific to quantised bases — bf16 bases don't need it.
# Skipping it on a quantised base causes NaN gradients in the first few steps.
model = prepare_model_for_kbit_training(model)

vram_gb = torch.cuda.memory_allocated(0) / 1024**3
print(f"Model loaded. VRAM: {vram_gb:.2f} GB  (target: ~4–5 GB for NF4)")


In [ ]:
# ── Cell 4: LoRA injection ────────────────────────────────────────────────────
from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGETS,
    # [CONCEPT] use_rslora
    # rslora scales alpha by sqrt(r) rather than alpha/r. This produces more
    # stable training at higher ranks (r=16+) because the effective scaling
    # decreases more slowly as r increases, preventing the adapter from
    # dominating the base weight at high rank values.
    use_rslora=True,
)
model = get_peft_model(model, lora_config)

# enable_input_require_grads() is needed for NF4 quantised bases where
# prepare_model_for_kbit_training() may not have set this automatically.
model.enable_input_require_grads()

model.print_trainable_parameters()
print(f"VRAM after LoRA injection: {torch.cuda.memory_allocated(0)/1024**3:.2f} GB")


## Section 3 — Training

Target: train loss < 0.80 (ADP-A is a generative task, not classification — loss
is higher than ADP-B/C). Monitor eval loss for overfitting.
Expected time on A10G: **20–40 minutes** for 5 epochs on ~700–900 records.


In [ ]:
# ── Cell 5: Training ─────────────────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

sft_config = SFTConfig(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    logging_steps=5,
    save_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    eval_strategy="steps",
    eval_steps=20,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",  # change to "wandb" if W&B is configured in Lightning.ai
    seed=42,
    data_seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    eval_dataset=eval_data,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)

steps_per_epoch = max(1, len(train_data) // (BATCH_SIZE * GRAD_ACCUM))
print(f"Steps/epoch : {steps_per_epoch}  |  Total steps : {steps_per_epoch * NUM_EPOCHS}")
print("Starting ADP-A training (Qwen3-4B QLoRA)...")

model.config.use_cache = False
train_result = trainer.train()

print("\n── Training complete ────────────────────────────────────────────────────")
print(f"  Final train loss : {train_result.training_loss:.4f}")
print(f"  Total steps      : {train_result.global_step}")
print(f"  Runtime          : {train_result.metrics.get('train_runtime',0)/60:.1f} min")
print(f"  Peak VRAM        : {torch.cuda.max_memory_allocated(0)/1024**3:.2f} GB")
if train_result.training_loss > 1.2:
    print("  WARN: Loss > 1.2 — review data quality in Step 20.")
else:
    print("  OK: Loss within expected range for generative empathy task.")


In [ ]:
# ── Cell 6: Save adapter + push to HF Hub ────────────────────────────────────
# [CONCEPT] Pushing to HF Hub
# trainer.model.save_pretrained() writes the adapter weights locally.
# push_to_hub() uploads them to the private HF Hub repo so the HF Space
# and Modal backend can pull them at startup.
# The base model (Qwen3-4B) is NOT re-uploaded — only the LoRA deltas (~25 MB).

trainer.model.save_pretrained(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)

adapter_mb = sum(f.stat().st_size for f in OUTPUT_DIR.rglob("*") if f.is_file()) / 1e6
print(f"Adapter saved locally: {OUTPUT_DIR.resolve()}")
print(f"Adapter size         : {adapter_mb:.1f} MB")

if HF_TOKEN and HF_OUTPUT_REPO:
    print(f"Pushing to HF Hub: {HF_OUTPUT_REPO}...")
    trainer.model.push_to_hub(HF_OUTPUT_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_OUTPUT_REPO, token=HF_TOKEN)
    print("Push complete.")
else:
    print("Skipping HF push (HF_TOKEN or HF_OUTPUT_REPO not set).")


In [ ]:
# ── Cell 7: Smoke test ───────────────────────────────────────────────────────
# Basic checks: model generates non-empty, sentence-capitalised, warm responses.
# The smoke test loads the saved adapter fresh to verify the save round-trip.
from peft import PeftModel

model.config.use_cache = True
model.gradient_checkpointing_disable()

TEST_PROMPTS = [
    "I've been feeling really anxious this week and I don't know why.",
    "i just feel like everything is pointless. like nothing matters.",  # paralinguistic
    "[Turn 1] User: I told my therapist I was doing better.\n[Turn 1] Nikko: What made things feel better?\n[Turn 2] User: I'm not sure it's real, honestly.",  # multi-turn
]

print("── ADP-A Smoke Test ─────────────────────────────────────────────────────")
for i, prompt in enumerate(TEST_PROMPTS):
    msgs = [{"role": "user", "content": prompt}]
    inp_str = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(inp_str, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = trainer.model.generate(**inputs, max_new_tokens=120, do_sample=False,
                                      temperature=1.0, pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

    starts_cap = resp and resp[0].isupper()   # typography rule
    non_empty  = len(resp.split()) >= 10
    no_url     = "http" not in resp and "@" not in resp

    status = "PASS" if (starts_cap and non_empty and no_url) else "FAIL"
    print(f"  T{i+1} [{status}] : {resp[:120]}...")
    if not starts_cap: print("         WARN: Response does not start with capital letter (typography rule).")
    if not non_empty:  print("         WARN: Response is too short.")
    if not no_url:     print("         WARN: Response contains URL or email (hallucination).")

print("\nStep 21 complete. ADP-A adapter ready at:", OUTPUT_DIR.resolve())
